# Punto 6: TSP Heurístico con Python

Resolución del Problema del Viajero (TSP) usando **algoritmos heurísticos** en Python puro.

Se implementan 5 heurísticas y se comparan contra la solución óptima (Held-Karp):
1. **Vecino más cercano** (Nearest Neighbor)
2. **2-opt** (mejora por intercambio de aristas)
3. **Christofides** (MST + matching + tour euleriano)
4. **Simulated Annealing** (recocido simulado)
5. **Algoritmo Genético** (población, cruce, mutación)

## 1. Carga de datos

In [ ]:
import json, time, random, math, os
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !wget -q -O ciudades.json https://raw.githubusercontent.com/d0ubt0/problema-del-viajero-bd2/main/shared/ciudades.json || True
    if not os.path.exists('ciudades.json'):
        print('Sube el archivo ciudades.json manualmente')
        uploaded = colab_files.upload()
        data = json.loads(uploaded[list(uploaded.keys())[0]])
    else:
        with open('ciudades.json') as f:
            data = json.load(f)
else:
    with open('../shared/ciudades.json') as f:
        data = json.load(f)

ciudades = data['ciudades']
dist_raw = data['distancias_km']
subsets = data['subsets']
n_total = len(ciudades)

dist_full = [[0]*n_total for _ in range(n_total)]
for key, val in dist_raw.items():
    a, b = key.split('-')
    a, b = int(a), int(b)
    dist_full[a][b] = val
    dist_full[b][a] = val

nombres = [c['nombre'] for c in ciudades]
coords = [(c['lat'], c['lon']) for c in ciudades]

print(f'{n_total} ciudades cargadas')
for c in ciudades:
    print(f"  {c['id']:>2}: {c['nombre']:<15} ({c['lat']}, {c['lon']})")

## 2. Funciones auxiliares

In [ ]:
def build_sub_matrix(dist_full, indices):
    n = len(indices)
    sub = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            sub[i][j] = dist_full[indices[i]][indices[j]]
    return sub

def tour_cost(tour, dist):
    return sum(dist[tour[i]][tour[i+1]] for i in range(len(tour)-1))

## 3. Held-Karp (solución óptima para comparación)

Algoritmo de programación dinámica con complejidad $O(n^2 \cdot 2^n)$. Solo factible para $n \leq 20$.

In [ ]:
def held_karp(dist, n):
    if n > 20:
        return None, float('inf')
    INF = float('inf')
    full_mask = (1 << n) - 1
    dp = [[INF]*n for _ in range(1 << n)]
    parent = [[-1]*n for _ in range(1 << n)]
    dp[1][0] = 0
    for mask in range(1, 1 << n):
        if not (mask & 1): continue
        for u in range(n):
            if not (mask & (1 << u)): continue
            if dp[mask][u] == INF: continue
            for v in range(n):
                if mask & (1 << v): continue
                new_mask = mask | (1 << v)
                new_cost = dp[mask][u] + dist[u][v]
                if new_cost < dp[new_mask][v]:
                    dp[new_mask][v] = new_cost
                    parent[new_mask][v] = u
    best_cost = INF
    last_city = -1
    for u in range(1, n):
        cost = dp[full_mask][u] + dist[u][0]
        if cost < best_cost:
            best_cost = cost
            last_city = u
    if best_cost == INF:
        return None, INF
    route = []
    mask = full_mask
    curr = last_city
    while curr != -1:
        route.append(curr)
        prev = parent[mask][curr]
        mask = mask ^ (1 << curr)
        curr = prev
    route.reverse()
    route.append(0)
    return route, best_cost

## 4. Heurística 1: Vecino más cercano

Algoritmo constructivo greedy: desde cada ciudad de inicio, siempre ir a la ciudad no visitada más cercana. Se toma el mejor resultado entre todas las ciudades de inicio.

In [ ]:
def nearest_neighbor(dist, n):
    best_tour = None
    best_cost = float('inf')
    for start in range(n):
        visited = [False]*n
        tour = [start]
        visited[start] = True
        for _ in range(n-1):
            curr = tour[-1]
            nearest = -1
            nearest_d = float('inf')
            for j in range(n):
                if not visited[j] and dist[curr][j] < nearest_d:
                    nearest_d = dist[curr][j]
                    nearest = j
            tour.append(nearest)
            visited[nearest] = True
        tour.append(start)
        c = tour_cost(tour, dist)
        if c < best_cost:
            best_cost = c
            best_tour = tour[:]
    return best_tour, best_cost

## 5. Heurística 2: 2-opt

Mejora iterativa: parte de un tour aleatorio (o del vecino más cercano) e intercambia pares de aristas mientras se encuentren mejoras.

In [ ]:
def two_opt(dist, n, initial_tour=None, max_iterations=1000):
    if initial_tour is None:
        tour = list(range(n))
        random.shuffle(tour)
        tour.append(tour[0])
    else:
        tour = initial_tour[:]
    improved = True
    iterations = 0
    while improved and iterations < max_iterations:
        improved = False
        iterations += 1
        for i in range(1, len(tour)-2):
            for j in range(i+1, len(tour)-1):
                d1 = dist[tour[i-1]][tour[i]] + dist[tour[j]][tour[j+1]]
                d2 = dist[tour[i-1]][tour[j]] + dist[tour[i]][tour[j+1]]
                if d2 < d1:
                    tour[i:j+1] = reversed(tour[i:j+1])
                    improved = True
    return tour, tour_cost(tour, dist)

def two_opt_multi(dist, n, runs=10):
    best_tour = None
    best_cost = float('inf')
    for _ in range(runs):
        tour, cost = two_opt(dist, n)
        if cost < best_cost:
            best_cost = cost
            best_tour = tour[:]
    nn_tour, nn_cost = nearest_neighbor(dist, n)
    tour, cost = two_opt(dist, n, initial_tour=nn_tour)
    if cost < best_cost:
        best_cost = cost
        best_tour = tour[:]
    return best_tour, best_cost

## 6. Heurística 3: Christofides

Algoritmo constructivo basado en:
1. Construir el árbol de expansión mínima (MST)
2. Encontrar matching mínimo sobre vértices de grado impar
3. Combinar en grafo euleriano
4. Obtener tour euleriano y aplicar shortcut para Hamiltoniano

Garantiza un factor de aproximación de 3/2 para TSP métrico.

In [ ]:
def prim_mst(dist, n):
    in_tree = [False]*n
    min_edge = [float('inf')]*n
    parent = [-1]*n
    min_edge[0] = 0
    edges = []
    for _ in range(n):
        u = -1
        for v in range(n):
            if not in_tree[v] and (u == -1 or min_edge[v] < min_edge[u]):
                u = v
        in_tree[u] = True
        if parent[u] != -1:
            edges.append((parent[u], u, dist[parent[u]][u]))
        for v in range(n):
            if not in_tree[v] and dist[u][v] < min_edge[v]:
                min_edge[v] = dist[u][v]
                parent[v] = u
    return edges

def min_weight_matching(adj, odd_vertices, dist):
    edges = []
    used = set()
    pairs = list(combinations(odd_vertices, 2))
    pairs.sort(key=lambda p: dist[p[0]][p[1]])
    for u, v in pairs:
        if u not in used and v not in used:
            edges.append((u, v, dist[u][v]))
            used.add(u)
            used.add(v)
            if len(used) == len(odd_vertices):
                break
    return edges

def euler_tour(adj, start, n):
    tour = []
    stack = [start]
    edge_count = {}
    for u in adj:
        for v in adj[u]:
            key = (min(u,v), max(u,v))
            edge_count[key] = edge_count.get(key, 0) + 1
    while stack:
        u = stack[-1]
        found = False
        for v in adj[u]:
            key = (min(u,v), max(u,v))
            if edge_count.get(key, 0) > 0:
                edge_count[key] -= 1
                adj[u].remove(v)
                adj[v].remove(u)
                stack.append(v)
                found = True
                break
        if not found:
            tour.append(stack.pop())
    tour.reverse()
    return tour

def christofides(dist, n):
    if n <= 2:
        tour = list(range(n)) + [0]
        return tour, tour_cost(tour, dist)
    mst_edges = prim_mst(dist, n)
    degree = [0]*n
    adj = {i: [] for i in range(n)}
    for u, v, w in mst_edges:
        adj[u].append(v)
        adj[v].append(u)
        degree[u] += 1
        degree[v] += 1
    odd_vertices = [i for i in range(n) if degree[i] % 2 == 1]
    matching_edges = min_weight_matching(adj, odd_vertices, dist)
    for u, v, w in matching_edges:
        adj[u].append(v)
        adj[v].append(u)
    euler = euler_tour(adj, 0, n)
    visited = set()
    tour = []
    for node in euler:
        if node not in visited:
            tour.append(node)
            visited.add(node)
    tour.append(tour[0])
    return tour, tour_cost(tour, dist)

## 7. Heurística 4: Simulated Annealing

Metaheurística inspirada en el proceso de recocido en metalurgia. Acepta soluciones peores con probabilidad decreciente según la temperatura baja.

In [ ]:
def simulated_annealing(dist, n, initial_temp=10000, cooling_rate=0.9995, min_temp=0.1):
    tour = list(range(n))
    random.shuffle(tour)
    tour.append(tour[0])
    current_cost = tour_cost(tour, dist)
    best_tour = tour[:]
    best_cost = current_cost
    temp = initial_temp
    while temp > min_temp:
        i = random.randint(1, n-1)
        j = random.randint(1, n-1)
        if i > j: i, j = j, i
        new_tour = tour[:]
        new_tour[i:j+1] = reversed(new_tour[i:j+1])
        new_cost = tour_cost(new_tour, dist)
        delta = new_cost - current_cost
        if delta < 0 or random.random() < math.exp(-delta / temp):
            tour = new_tour
            current_cost = new_cost
            if current_cost < best_cost:
                best_cost = current_cost
                best_tour = tour[:]
        temp *= cooling_rate
    return best_tour, best_cost

## 8. Heurística 5: Algoritmo Genético

Población de tours que evoluciona mediante selección por torneo, cruce ordenado (OX) y mutación por intercambio.

In [ ]:
def ga_crossover(parent1, parent2, n):
    start = random.randint(0, n-1)
    end = random.randint(start+1, n)
    child = [-1]*n
    p1 = parent1[:-1]
    p2 = parent2[:-1]
    child[start:end] = p1[start:end]
    used = set(child[start:end])
    pos = end % n
    for gene in p2:
        if gene not in used:
            child[pos] = gene
            used.add(gene)
            pos = (pos + 1) % n
    child.append(child[0])
    return child

def ga_mutate(tour, n, mutation_rate=0.3):
    if random.random() < mutation_rate:
        t = tour[:-1]
        i = random.randint(0, n-1)
        j = random.randint(0, n-1)
        t[i], t[j] = t[j], t[i]
        t.append(t[0])
        return t
    return tour

def genetic_algorithm(dist, n, population_size=100, generations=300, mutation_rate=0.3):
    population = []
    for _ in range(population_size):
        tour = list(range(n))
        random.shuffle(tour)
        tour.append(tour[0])
        population.append(tour)
    best_tour = None
    best_cost = float('inf')
    for gen in range(generations):
        fitness = [(tour_cost(t, dist), t) for t in population]
        fitness.sort(key=lambda x: x[0])
        if fitness[0][0] < best_cost:
            best_cost = fitness[0][0]
            best_tour = fitness[0][1][:]
        elite_count = max(2, population_size // 10)
        new_population = [f[1][:] for f in fitness[:elite_count]]
        while len(new_population) < population_size:
            tournament_size = 5
            candidates = random.sample(fitness, min(tournament_size, len(fitness)))
            candidates.sort(key=lambda x: x[0])
            p1 = candidates[0][1]
            candidates2 = random.sample(fitness, min(tournament_size, len(fitness)))
            candidates2.sort(key=lambda x: x[0])
            p2 = candidates2[0][1]
            child = ga_crossover(p1, p2, n)
            child = ga_mutate(child, n, mutation_rate)
            new_population.append(child)
        population = new_population
    return best_tour, best_cost

## 9. Ejecución de benchmarks

In [ ]:
random.seed(42)

def run_heuristic(func, dist, n, *args, **kwargs):
    t0 = time.perf_counter()
    tour, cost = func(dist, n, *args, **kwargs)
    t1 = time.perf_counter()
    return tour, cost, (t1 - t0) * 1000

sizes = [4, 6, 8, 10, 12, 14, 16, 18, 20]
all_results = []

for size in sizes:
    key = str(size)
    if key not in subsets: continue
    indices = subsets[key]
    sub_dist = build_sub_matrix(dist_full, indices)
    sub_nombres = [nombres[i] for i in indices]

    optimal_cost = None
    optimal_time = None
    if size <= 20:
        try:
            t0 = time.perf_counter()
            hk_route, hk_cost = held_karp(sub_dist, size)
            t1 = time.perf_counter()
            optimal_time = (t1 - t0) * 1000
            if hk_route is not None:
                optimal_cost = hk_cost
        except: pass

    nn_tour, nn_cost, nn_time = run_heuristic(nearest_neighbor, sub_dist, size)
    to_tour, to_cost, to_time = run_heuristic(two_opt_multi, sub_dist, size)
    ch_tour, ch_cost, ch_time = run_heuristic(christofides, sub_dist, size)
    sa_tour, sa_cost, sa_time = run_heuristic(simulated_annealing, sub_dist, size)
    ga_tour, ga_cost, ga_time = run_heuristic(genetic_algorithm, sub_dist, size)

    def dev(cost):
        if optimal_cost is not None and optimal_cost > 0:
            return ((cost - optimal_cost) / optimal_cost) * 100
        return None

    print(f'\n{"="*80}')
    print(f'=== {size} ciudades: {", ".join(sub_nombres)} ===')
    print(f'{"="*80}')
    print(f'{"Algoritmo":<28} | {"Distancia":>10} | {"Tiempo (ms)":>12} | {"Desviación (%)":>14}')
    print('-' * 72)
    if optimal_cost is not None:
        print(f'{"Óptimo (Held-Karp)":<28} | {optimal_cost:>10} | {optimal_time:>12.2f} | {"0.00":>14}')
    algos = [
        ('Vecino más cercano', nn_cost, nn_time),
        ('2-opt', to_cost, to_time),
        ('Christofides', ch_cost, ch_time),
        ('Simulated Annealing', sa_cost, sa_time),
        ('Genético', ga_cost, ga_time),
    ]
    for name, cost, t in algos:
        d = dev(cost)
        d_str = f'{d:.2f}' if d is not None else 'N/A'
        print(f'{name:<28} | {cost:>10} | {t:>12.2f} | {d_str:>14}')

    all_results.append({
        'size': size,
        'optimal_cost': optimal_cost,
        'optimal_time': optimal_time,
        'nn': {'cost': nn_cost, 'time': nn_time, 'dev': dev(nn_cost), 'tour': nn_tour},
        'two_opt': {'cost': to_cost, 'time': to_time, 'dev': dev(to_cost), 'tour': to_tour},
        'christofides': {'cost': ch_cost, 'time': ch_time, 'dev': dev(ch_cost), 'tour': ch_tour},
        'sa': {'cost': sa_cost, 'time': sa_time, 'dev': dev(sa_cost), 'tour': sa_tour},
        'ga': {'cost': ga_cost, 'time': ga_time, 'dev': dev(ga_cost), 'tour': ga_tour},
        'indices': indices,
    })

## 10. Visualización de rutas

In [ ]:
def plot_route(indices, tour, sub_coords, title, ax):
    lats = [sub_coords[tour[i]][0] for i in range(len(tour))]
    lons = [sub_coords[tour[i]][1] for i in range(len(tour))]
    ax.plot(lons, lats, 'b-', linewidth=1.5, alpha=0.7)
    ax.plot(lons, lats, 'ro', markersize=6)
    ax.plot(lons[0], lats[0], 'g^', markersize=10, label='Inicio')
    for i, idx in enumerate(tour[:-1]):
        ax.annotate(str(i), (lons[i], lats[i]), fontsize=7, ha='right')
    ax.set_title(title)
    ax.set_xlabel('Longitud')
    ax.set_ylabel('Latitud')
    ax.grid(True, alpha=0.3)

for r in all_results[:4]:
    size = r['size']
    indices = r['indices']
    sub_coords = [coords[i] for i in indices]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Rutas para {size} ciudades', fontsize=16, fontweight='bold')
    algo_list = [
        ('nn', 'Vecino más cercano'),
        ('two_opt', '2-opt'),
        ('christofides', 'Christofides'),
        ('sa', 'Simulated Annealing'),
        ('ga', 'Genético'),
    ]
    for idx, (key, name) in enumerate(algo_list):
        ax = axes[idx // 3][idx % 3]
        tour = r[key]['tour']
        cost = r[key]['cost']
        d = r[key]['dev']
        d_str = f'{d:.1f}%' if d is not None else 'N/A'
        plot_route(indices, tour, sub_coords, f'{name}\n{cost} km (dev: {d_str})', ax)
    axes[1][2].axis('off')
    plt.tight_layout()
    plt.show()

## 11. Gráficos comparativos

In [ ]:
sizes_list = [r['size'] for r in all_results]
opt_costs = [r['optimal_cost'] if r['optimal_cost'] else 0 for r in all_results]
nn_costs = [r['nn']['cost'] for r in all_results]
to_costs = [r['two_opt']['cost'] for r in all_results]
ch_costs = [r['christofides']['cost'] for r in all_results]
sa_costs = [r['sa']['cost'] for r in all_results]
ga_costs = [r['ga']['cost'] for r in all_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(sizes_list))
width = 0.12
ax1.bar(x - 2*width, opt_costs, width, label='Óptimo', color='black')
ax1.bar(x - width, nn_costs, width, label='Vecino', color='blue')
ax1.bar(x, to_costs, width, label='2-opt', color='green')
ax1.bar(x + width, ch_costs, width, label='Christofides', color='orange')
ax1.bar(x + 2*width, sa_costs, width, label='SA', color='red')
ax1.bar(x + 3*width, ga_costs, width, label='Genético', color='purple')
ax1.set_xlabel('Número de ciudades')
ax1.set_ylabel('Distancia (km)')
ax1.set_title('Distancias por algoritmo')
ax1.set_xticks(x)
ax1.set_xticklabels(sizes_list)
ax1.legend()
ax1.grid(True, alpha=0.3)

nn_times = [r['nn']['time'] for r in all_results]
to_times = [r['two_opt']['time'] for r in all_results]
ch_times = [r['christofides']['time'] for r in all_results]
sa_times = [r['sa']['time'] for r in all_results]
ga_times = [r['ga']['time'] for r in all_results]
opt_times = [r['optimal_time'] if r['optimal_time'] else 0 for r in all_results]

ax2.plot(sizes_list, opt_times, 'k-o', label='Óptimo')
ax2.plot(sizes_list, nn_times, 'b-s', label='Vecino')
ax2.plot(sizes_list, to_times, 'g-^', label='2-opt')
ax2.plot(sizes_list, ch_times, 'orange', marker='D', label='Christofides')
ax2.plot(sizes_list, sa_times, 'r-v', label='SA')
ax2.plot(sizes_list, ga_times, 'm-<', label='Genético')
ax2.set_xlabel('Número de ciudades')
ax2.set_ylabel('Tiempo (ms)')
ax2.set_title('Tiempo de ejecución')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
nn_devs = [r['nn']['dev'] if r['nn']['dev'] is not None else 0 for r in all_results]
to_devs = [r['two_opt']['dev'] if r['two_opt']['dev'] is not None else 0 for r in all_results]
ch_devs = [r['christofides']['dev'] if r['christofides']['dev'] is not None else 0 for r in all_results]
sa_devs = [r['sa']['dev'] if r['sa']['dev'] is not None else 0 for r in all_results]
ga_devs = [r['ga']['dev'] if r['ga']['dev'] is not None else 0 for r in all_results]

ax.plot(sizes_list, nn_devs, 'b-o', label='Vecino')
ax.plot(sizes_list, to_devs, 'g-s', label='2-opt')
ax.plot(sizes_list, ch_devs, color='orange', marker='^', label='Christofides')
ax.plot(sizes_list, sa_devs, 'r-D', label='SA')
ax.plot(sizes_list, ga_devs, 'm-v', label='Genético')
ax.set_xlabel('Número de ciudades')
ax.set_ylabel('Desviación del óptimo (%)')
ax.set_title('Desviación porcentual respecto al óptimo')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Tabla resumen y análisis

In [ ]:
print('=' * 100)
print('TABLA RESUMEN')
print('=' * 100)
header = f'{"N":>3} | {"Óptimo":>8} | {"Vecino":>8} | {"2-opt":>8} | {"Christof.":>9} | {"SA":>8} | {"Genético":>8}'
print(header)
print('-' * len(header))
for r in all_results:
    opt = f"{r['optimal_cost']}" if r['optimal_cost'] is not None else 'N/A'
    print(f"{r['size']:>3} | {opt:>8} | {r['nn']['cost']:>8} | {r['two_opt']['cost']:>8} | {r['christofides']['cost']:>9} | {r['sa']['cost']:>8} | {r['ga']['cost']:>8}")

avg_devs = {}
for algo_name, key in [('Vecino','nn'),('2-opt','two_opt'),('Christofides','christofides'),('SA','sa'),('Genético','ga')]:
    devs = [r[key]['dev'] for r in all_results if r[key]['dev'] is not None]
    if devs:
        avg_devs[algo_name] = sum(devs)/len(devs)

print('\nDesviación promedio del óptimo:')
for name, avg in sorted(avg_devs.items(), key=lambda x: x[1]):
    print(f'  {name:<20}: {avg:.2f}%')
if avg_devs:
    best = min(avg_devs, key=avg_devs.get)
    print(f'\n  >>> En promedio, "{best}" es la heurística más cercana al óptimo.')